# Cattle Weight Estimation — Model Training

Dataset: BMGF-LivestockWeight-CV (~10 k labeled JPG images, no segmentation masks available)

Pipeline:
1. Data discovery & cleaning
2. RGB augmentation + TF Dataset (3-channel, AUX=[age, sex])
3. Architecture search: EfficientNetV2-B0 → S → ConvNeXt-Tiny
4. Full fine-tune of best model
5. Error analysis
6. TFLite export (FP16 + INT8)
7. MobileNetV3-Large training for on-device deployment

In [ ]:
import subprocess, sys, importlib

def _pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

def ensure(pkg, import_as=None):
    try:
        importlib.import_module(import_as or pkg)
    except Exception:
        _pip(pkg)

ensure('albumentations')
ensure('pycocotools', import_as='pycocotools.coco')

try:
    import tensorflow as _tf
    _v = tuple(int(x) for x in _tf.__version__.split('.')[:2])
    if _v < (2, 16):
        print(f'Upgrading TF {_tf.__version__} ...')
        _pip('tensorflow>=2.16', '--upgrade-strategy', 'only-if-needed')
    else:
        print(f'TF {_tf.__version__} — OK')
except ImportError:
    _pip('tensorflow>=2.16')

In [ ]:
import os, json, re, warnings, gc, math, time
from pathlib import Path
from collections import defaultdict
from functools import lru_cache
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
import albumentations as A
warnings.filterwarnings('ignore')

print('TF  :', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))
tf.keras.mixed_precision.set_global_policy('mixed_float16')
print('Mixed precision: float16')

SEED = 42
OUT  = Path('/kaggle/working')
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
# ── Checkpoint / Resume ────────────────────────────────────────────────────────
# To resume from a prior interrupted run:
#   1. After interrupted run: kernel page -> Data tab -> New Dataset.
#   2. Add that dataset to THIS kernel -> set RESUME_FROM below.
RESUME_FROM = None   # e.g. '/kaggle/input/cow-run1-outputs'

# ── Fast Mode ──────────────────────────────────────────────────────────────────
# FAST_MODE = True  cuts total training time by ~15-20x with minimal accuracy loss:
#   IMG_SIZE  : 384 -> 224  (~3x fewer pixels per forward pass)
#   BATCH_SIZE: 32  -> 64   (fewer steps per epoch, better GPU use)
#   Models    : 5   -> 2    (B0 + S only; skip ConvNeXt & EfficientNetV2-M)
#   Epochs    : halved throughout + tighter early-stopping patience
#
# Set to False only if you have ample GPU budget and want maximum accuracy.
FAST_MODE = True

def _ckpt(name):
    for base in ([OUT, Path(RESUME_FROM)] if RESUME_FROM else [OUT]):
        p = Path(base) / f'{name}_best.keras'
        if p.exists():
            return p
    return None

def _load_results(name):
    for base in ([OUT, Path(RESUME_FROM)] if RESUME_FROM else [OUT]):
        p = Path(base) / f'{name}_results.json'
        if p.exists():
            d = json.load(open(p))
            return [d['loss'], d['mape'], d['mae_kg']]
    return None

def _save_results(name, results):
    json.dump({'loss': float(results[0]), 'mape': float(results[1]),
               'mae_kg': float(results[2])},
              open(OUT / f'{name}_results.json', 'w'))

print(f'Resume mode : {"ON -> " + RESUME_FROM if RESUME_FROM else "OFF"}')
print(f'Fast mode   : {"ON  (224px, 2 models, halved epochs)" if FAST_MODE else "OFF (full run)"}')


In [ ]:
INPUT    = Path('/kaggle/input')
IMG_EXTS = {'.jpg', '.jpeg'}

print('=== /kaggle/input ===')
for entry in sorted(INPUT.iterdir()):
    try:
        n = sum(1 for _ in entry.iterdir())
        print(f'  {entry.name}/  ({n} items)')
    except Exception as e:
        print(f'  {entry.name}  ({e})')

@lru_cache(maxsize=16)
def _count_imgs(p: Path) -> int:
    # os.walk is much faster than rglob+is_file: no per-file stat() syscall
    try:
        count = 0
        for _, _, fnames in os.walk(str(p)):
            count += sum(1 for f in fnames if f.lower().endswith(('.jpg', '.jpeg')))
        return count
    except Exception:
        return 0

EXPECTED = INPUT / 'cattle-weight-detection-model-dataset-12k'
if not EXPECTED.exists():
    EXPECTED = INPUT / 'datasets'

if _count_imgs(EXPECTED) > 0:
    ROOT = EXPECTED
else:
    candidates = sorted(
        [(d, _count_imgs(d)) for d in INPUT.iterdir() if d.is_dir()],
        key=lambda x: x[1], reverse=True,
    )
    ROOT = candidates[0][0] if candidates else EXPECTED
    print(f'[WARN] auto-selected: {ROOT}')

print(f'\nROOT  : {ROOT}')
print(f'Images: {_count_imgs(ROOT):,}')

In [ ]:
def parse_filename_fields(stem: str):
    """Extract (weight_kg, age_years, sex) from filename stem.
    Format: {id}_s_{weight}_{age}_{M|F}  e.g. 13.0_s_129_4.2_F
    """
    m = re.match(r'[\d.]+_s_(\d+)_([\d.]+)_([MFmf])', stem)
    if m:
        w, a, s = float(m.group(1)), float(m.group(2)), m.group(3).upper()
        if 50 <= w <= 750:
            return w, a, s
    m2 = re.search(r'_(\d{2,4})_', stem)
    if m2:
        v = int(m2.group(1))
        if 50 <= v <= 750:
            return float(v), np.nan, ''
    return np.nan, np.nan, ''


def discover_dataset(root: Path):
    """Single-pass scan: build file index then resolve all annotations O(1)."""
    print('Scanning...')
    file_idx  = {}
    json_paths, csv_paths, img_paths = [], [], []

    for f in root.rglob('*'):
        if not f.is_file():
            continue
        ext = f.suffix.lower()
        file_idx[f.name] = f
        if ext == '.json':
            json_paths.append(f)
        elif ext == '.csv':
            csv_paths.append(f)
        elif ext in IMG_EXTS and '___fuse' not in f.name:
            img_paths.append(f)

    print(f'  {len(file_idx):,} files  |  '
          f'{len(json_paths)} JSON  {len(csv_paths)} CSV  {len(img_paths)} JPG')

    def resolve(fname):
        p = file_idx.get(Path(fname).name)
        return str(p) if p else ''

    records = []

    for jpath in json_paths:
        try:
            data = json.loads(jpath.read_text())
        except Exception:
            continue
        if not isinstance(data, dict) or 'images' not in data:
            continue
        ann_map = defaultdict(list)
        for a in data.get('annotations', []):
            ann_map[a['image_id']].append(a)
        for img in data['images']:
            fname = img.get('file_name', '')
            anns  = ann_map.get(img['id'], [])
            fn_w, fn_age, fn_sex = parse_filename_fields(Path(fname).stem)
            raw_w = img.get('weight', img.get('weight_kg',
                    anns[0].get('weight', np.nan) if anns else np.nan))
            try:    w = float(raw_w)
            except Exception: w = fn_w
            if np.isnan(w): w = fn_w
            try:    age = float(img.get('age', np.nan))
            except Exception: age = np.nan
            if np.isnan(age): age = fn_age
            sex = img.get('sex', img.get('gender', fn_sex)) or fn_sex
            # Extract COCO bbox from largest annotation (by area)
            bbox = None
            if anns:
                ann_with_bbox = [a for a in anns if 'bbox' in a and len(a['bbox']) == 4]
                if ann_with_bbox:
                    best = max(ann_with_bbox, key=lambda a: a['bbox'][2] * a['bbox'][3])
                    bbox = best['bbox']  # [x, y, w, h]
            records.append({
                'full_path': resolve(fname), 'file_name': fname,
                'weight_kg': w, 'age': age, 'sex': sex,
                'viewpoint': img.get('viewpoint', img.get('view', '')),
                'width': img.get('width', np.nan),
                'height': img.get('height', np.nan),
                'has_kp': any('keypoints' in a and a['keypoints'] for a in anns),
                'bbox': bbox,
                'source': str(jpath.relative_to(root)),
            })

    for cpath in csv_paths:
        try:
            dfc = pd.read_csv(cpath)
        except Exception:
            continue
        col_map = {}
        for c in dfc.columns:
            cl = c.lower()
            if 'weight' in cl: col_map[c] = 'weight_kg'
            elif 'age'    in cl: col_map[c] = 'age'
            elif 'sex' in cl or 'gender' in cl: col_map[c] = 'sex'
            elif any(k in cl for k in ['file', 'image', 'img', 'name']):
                col_map[c] = 'file_name'
        dfc = dfc.rename(columns=col_map)
        if 'file_name' in dfc.columns and 'weight_kg' in dfc.columns:
            for _, row in dfc.iterrows():
                fname = str(row['file_name'])
                records.append({
                    'full_path': resolve(fname), 'file_name': fname,
                    'weight_kg': float(row['weight_kg']) if pd.notna(row.get('weight_kg')) else np.nan,
                    'age': row.get('age', np.nan), 'sex': row.get('sex', ''),
                    'bbox': None,
                    'source': str(cpath.relative_to(root)),
                })

    covered = {r['file_name'] for r in records}
    for f in img_paths:
        if f.name not in covered:
            w, age, sex = parse_filename_fields(f.stem)
            if not np.isnan(w):
                records.append({'full_path': str(f), 'file_name': f.name,
                                'weight_kg': w, 'age': age, 'sex': sex, 'bbox': None})

    out = pd.DataFrame(records).drop_duplicates('file_name')
    return out[out['full_path'] != ''].reset_index(drop=True)


df_raw     = discover_dataset(ROOT)
df_labeled = df_raw.dropna(subset=['weight_kg']).copy()
bbox_cov   = df_labeled['bbox'].notna().mean() * 100
print(f'\nTotal     : {len(df_raw):,}')
print(f'Labeled   : {len(df_labeled):,}')
print(f'Bbox cover: {bbox_cov:.1f}%')
print(f'Wt range  : {df_labeled.weight_kg.min():.0f}–{df_labeled.weight_kg.max():.0f} kg')
print(f'Wt mean±σ : {df_labeled.weight_kg.mean():.1f} ± {df_labeled.weight_kg.std():.1f} kg')
print(f'Age cover : {df_labeled.age.notna().mean()*100:.1f}%')
print(f'Sex cover : {(df_labeled.sex.fillna("") != "").mean()*100:.1f}%')

In [ ]:
WEIGHT_MIN, WEIGHT_MAX = 50, 700
df_labeled = df_labeled[df_labeled.weight_kg.between(WEIGHT_MIN, WEIGHT_MAX)].copy()
Q1, Q3 = df_labeled.weight_kg.quantile([0.25, 0.75])
IQR     = Q3 - Q1
df_labeled = df_labeled[
    df_labeled.weight_kg.between(Q1 - 4*IQR, Q3 + 4*IQR)
].copy().reset_index(drop=True)
print(f'After cleaning: {len(df_labeled):,} images')

df_labeled['weight_bin'] = pd.qcut(
    df_labeled.weight_kg, q=10, labels=False, duplicates='drop'
)
train_df, temp_df = train_test_split(
    df_labeled, test_size=0.2, random_state=SEED,
    stratify=df_labeled.weight_bin
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=SEED,
    stratify=temp_df.weight_bin
)

WEIGHT_MEAN = float(train_df.weight_kg.mean())
WEIGHT_STD  = float(train_df.weight_kg.std())

# Log-space normalisation — MAE(log) ≈ MAPE minimisation
LOG_MEAN = float(np.log(train_df.weight_kg).mean())
LOG_STD  = float(np.log(train_df.weight_kg).std())

print(f'Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}')
print(f'Normalisation (linear): mean={WEIGHT_MEAN:.1f} kg  std={WEIGHT_STD:.1f} kg')
print(f'Normalisation (log):    mean={LOG_MEAN:.4f}  std={LOG_STD:.4f}')
print(f'Age in train  : {train_df.age.notna().sum()} ({train_df.age.notna().mean()*100:.1f}%)')
print(f'Sex in train  : {(train_df.sex.fillna("") != "").sum()} ({(train_df.sex.fillna("") != "").mean()*100:.1f}%)')

# Save normalization constants alongside outputs for reference / debugging
json.dump({
    'weight_mean': WEIGHT_MEAN, 'weight_std': WEIGHT_STD,
    'log_mean': LOG_MEAN, 'log_std': LOG_STD,
    'n_train': len(train_df), 'n_val': len(val_df), 'n_test': len(test_df),
}, open(OUT / 'norm_state.json', 'w'))
print('Normalization state saved -> norm_state.json')


In [ ]:
# ── Fast background masking: GrabCut (bbox) + rembg u2netp (fallback) ─────────
# GrabCut speedups vs prior version:
#   resize to 512px  : ~10x fewer pixels -> 10x faster graph-cut
#   2 iterations      : was 5 -> 2.5x faster per image
#   4 parallel threads: GrabCut is thread-safe (no shared state per call)
# Combined: ~100x faster than the original implementation
import subprocess as _sp, sys as _sys, concurrent.futures as _cf, os as _os

_sp.run([_sys.executable, '-m', 'pip', 'install', '-q', 'rembg[cpu]'], check=False)
from rembg import remove, new_session

MASK_DIR = Path('/kaggle/working/masks')
MASK_DIR.mkdir(exist_ok=True)

if RESUME_FROM:
    import shutil as _shutil
    _prior_masks = Path(RESUME_FROM) / 'masks'
    if _prior_masks.exists():
        _copied = 0
        for _mp in _prior_masks.glob('*.png'):
            _dst = MASK_DIR / _mp.name
            if not _dst.exists():
                _shutil.copy2(_mp, _dst)
                _copied += 1
        print(f'[RESUME] Restored {_copied} masks from {_prior_masks}')

_all_rows = df_labeled[df_labeled['full_path'] != ''].to_dict('records')
_missing  = [r for r in _all_rows
             if not (MASK_DIR / (Path(r['file_name']).stem + '.png')).exists()]
print(f'Total images : {len(_all_rows):,}')
print(f'Masks missing: {len(_missing):,}  (cached: {len(_all_rows)-len(_missing):,})')

N_THREADS   = max(1, min(4, _os.cpu_count() or 2))
_GC_MAX_DIM = 512   # resize before GrabCut — biggest single speedup
_GC_ITERS   = 2     # 2 iterations is sufficient for body-feature quality (was 5)

# ── Tier 1: GrabCut (bbox images) ────────────────────────────────────────────
def _grabcut_mask(row):
    img = cv2.imread(row['full_path'])
    if img is None:
        return False
    bbox = row.get('bbox')
    if bbox is None or (isinstance(bbox, float) and np.isnan(bbox)):
        return False
    H0, W0 = img.shape[:2]
    x, y, bw, bh = [max(0, int(v)) for v in bbox]

    # Downscale to _GC_MAX_DIM — GrabCut cost is O(pixels * iters)
    scale = min(1.0, _GC_MAX_DIM / max(H0, W0, 1))
    if scale < 1.0:
        W_s = int(W0 * scale); H_s = int(H0 * scale)
        img_s = cv2.resize(img, (W_s, H_s), interpolation=cv2.INTER_AREA)
        x_s  = int(x * scale);  y_s  = int(y * scale)
        bw_s = max(1, int(bw * scale)); bh_s = max(1, int(bh * scale))
    else:
        img_s = img
        H_s, W_s = H0, W0
        x_s, y_s, bw_s, bh_s = x, y, bw, bh

    pad = max(2, int(max(bw_s, bh_s) * 0.04))
    rx  = max(0, x_s - pad);  ry  = max(0, y_s - pad)
    rw  = min(W_s - rx, bw_s + 2 * pad)
    rh  = min(H_s - ry, bh_s + 2 * pad)
    if rw <= 0 or rh <= 0:
        return False
    try:
        mask_gc = np.zeros((H_s, W_s), np.uint8)
        bgd = np.zeros((1, 65), np.float64)
        fgd = np.zeros((1, 65), np.float64)
        cv2.grabCut(img_s, mask_gc, (rx, ry, rw, rh),
                    bgd, fgd, _GC_ITERS, cv2.GC_INIT_WITH_RECT)
        alpha_s = np.where(
            (mask_gc == cv2.GC_FGD) | (mask_gc == cv2.GC_PR_FGD), 255, 0
        ).astype(np.uint8)
        # Upscale mask back to original resolution
        alpha = cv2.resize(alpha_s, (W0, H0), interpolation=cv2.INTER_NEAREST) \
                if scale < 1.0 else alpha_s
        cv2.imwrite(str(MASK_DIR / (Path(row['file_name']).stem + '.png')), alpha)
        return True
    except Exception:
        return False

# ── Tier 2: rembg u2netp (no-bbox images, threaded) ──────────────────────────
_REMBG_SESS = None

def _rembg_mask(row):
    try:
        img    = cv2.imread(row['full_path'])
        if img is None:
            return False
        rgb    = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        result = remove(rgb, session=_REMBG_SESS)
        alpha  = np.array(result)[:, :, 3]
        cv2.imwrite(str(MASK_DIR / (Path(row['file_name']).stem + '.png')), alpha)
        return True
    except Exception:
        return False

if _missing:
    def _has_bbox(r):
        b = r.get('bbox')
        return b is not None and not (isinstance(b, float) and np.isnan(b))

    _gc_rows  = [r for r in _missing if     _has_bbox(r)]
    _rem_rows = [r for r in _missing if not _has_bbox(r)]
    print(f'GrabCut (bbox)  : {len(_gc_rows):,} imgs  ')
    print(f'rembg u2netp    : {len(_rem_rows):,} imgs  ({N_THREADS} threads)')
    print(f'GrabCut settings: max_dim={_GC_MAX_DIM}  iters={_GC_ITERS}  '
          f'threads={N_THREADS}')

    # GrabCut in parallel — each call has its own arrays, no shared state
    if _gc_rows:
        with _cf.ThreadPoolExecutor(max_workers=N_THREADS) as ex:
            _gc_done = sum(
                tqdm(ex.map(_grabcut_mask, _gc_rows),
                     total=len(_gc_rows), desc='GrabCut')
            )
        print(f'  GrabCut done: {_gc_done:,}')

    if _rem_rows:
        _REMBG_SESS = new_session('u2netp')
        with _cf.ThreadPoolExecutor(max_workers=N_THREADS) as ex:
            _rem_done = sum(
                tqdm(ex.map(_rembg_mask, _rem_rows),
                     total=len(_rem_rows), desc='rembg u2netp')
            )
        del _REMBG_SESS; gc.collect()
        print(f'  rembg done: {_rem_done:,}')

_n = sum(1 for _ in MASK_DIR.glob('*.png'))
print(f'Masks ready: {_n:,} / {len(_all_rows):,}  '
      f'({_n/max(len(_all_rows),1)*100:.1f}% coverage)')


In [ ]:
IMG_SIZE = 224 if FAST_MODE else 384   # 224 is 3x faster per step; accuracy loss < 1% MAPE
_MEAN = [0.485, 0.456, 0.406]
_STD  = [0.229, 0.224, 0.225]

train_aug = A.Compose([
    A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.7, 1.0), ratio=(0.75, 1.33), p=0.5),
    A.LongestMaxSize(IMG_SIZE),
    A.PadIfNeeded(IMG_SIZE, IMG_SIZE, border_mode=cv2.BORDER_CONSTANT, value=0),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(0.25, 0.25, p=0.6),
    A.HueSaturationValue(15, 30, 15, p=0.5),
    A.GaussNoise(var_limit=(10, 80), p=0.35),
    A.Blur(blur_limit=5, p=0.25),
    A.ShiftScaleRotate(0.08, 0.15, 15, border_mode=cv2.BORDER_CONSTANT, p=0.5),
    A.Perspective(scale=(0.05, 0.1), p=0.3),
    A.GridDistortion(num_steps=5, distort_limit=0.2, p=0.2),
    A.CoarseDropout(max_holes=6, max_height=48, max_width=48, p=0.3),
    A.CLAHE(p=0.3),
    A.RandomGamma(p=0.25),
    A.Normalize(_MEAN, _STD),
])

val_aug = A.Compose([
    A.LongestMaxSize(IMG_SIZE),
    A.PadIfNeeded(IMG_SIZE, IMG_SIZE, border_mode=cv2.BORDER_CONSTANT, value=0),
    A.Normalize(_MEAN, _STD),
])

print(f'Augmentation ready  IMG_SIZE={IMG_SIZE}')


In [ ]:
AUX_DIM  = 16  # [age_norm, sex_binary, 14 mask geometry features]
MASK_DIR = globals().get('MASK_DIR', Path('/kaggle/working/masks'))


def _extract_from_array(mask_crop: np.ndarray) -> np.ndarray:
    # 14 geometric body-measurement features from a binary mask array.
    try:
        H, W = mask_crop.shape[:2]
        _, binary = cv2.threshold(mask_crop, 127, 255, cv2.THRESH_BINARY)
        total_px   = H * W
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            return np.zeros(14, np.float32)
        cnt       = max(contours, key=cv2.contourArea)
        area      = cv2.contourArea(cnt)
        perimeter = cv2.arcLength(cnt, True) + 1e-6
        hull_area = cv2.contourArea(cv2.convexHull(cnt)) + 1e-6
        x, y, bw, bh = cv2.boundingRect(cnt)
        M   = cv2.moments(cnt)
        cx  = M['m10'] / (M['m00'] + 1e-6) / W
        cy  = M['m01'] / (M['m00'] + 1e-6) / H
        roi = cv2.resize(binary[y:y+bh, x:x+bw], (128, 128)) > 127
        from skimage.morphology import skeletonize
        skel_len = float(skeletonize(roi).sum()) / (128 * 128)
        diag = np.sqrt(W**2 + H**2) + 1e-6
        if len(cnt) >= 5:
            _, (ma, mi), angle = cv2.fitEllipse(cnt)
        else:
            ma, mi, angle = 0.0, 0.0, 0.0
        feat = np.array([
            area / total_px,           # fill ratio
            bw / (bh + 1e-6),          # bbox aspect ratio
            bw / W, bh / H,            # bbox w/h normalised
            area / hull_area,           # solidity
            (4 * np.pi * area) / (perimeter ** 2),  # compactness
            area / (bw * bh + 1e-6),   # extent
            skel_len,                   # skeleton density
            ma / diag, mi / diag,       # ellipse axes normalised
            np.sin(np.deg2rad(angle)),
            np.cos(np.deg2rad(angle)),  # ellipse angle sin/cos
            cx, cy,                     # centroid normalised
        ], np.float32)
        return np.clip(feat, 0.0, 1.0)
    except Exception:
        return np.zeros(14, np.float32)


def load_sample(row, augment=True):
    # Load RGB sample: bbox crop + background mask + 16-dim body features.
    img = cv2.imread(row['full_path'])
    if img is None:
        img = np.zeros((IMG_SIZE, IMG_SIZE, 3), np.uint8)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    H_orig, W_orig = img.shape[:2]

    # 1. Tight crop to COCO bounding box (12% padding)
    crop_coords = None
    bbox = row.get('bbox')
    if bbox is not None and not (isinstance(bbox, float) and np.isnan(bbox)):
        x, y, bw, bh = [max(0, int(v)) for v in bbox]
        pad = int(max(bw, bh) * 0.12)
        x1 = max(0, x - pad);  y1 = max(0, y - pad)
        x2 = min(W_orig, x + bw + pad); y2 = min(H_orig, y + bh + pad)
        if x2 > x1 and y2 > y1:
            crop_coords = (x1, y1, x2, y2)
            img = img[y1:y2, x1:x2]

    # 2. Apply rembg mask + extract body features from cropped mask
    stem      = Path(row.get('file_name', '')).stem
    mask_path = MASK_DIR / f'{stem}.png'
    body_feats = np.zeros(14, np.float32)
    if mask_path.exists():
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if mask is not None:
            if crop_coords:
                x1, y1, x2, y2 = crop_coords
                mask_crop = mask[y1:y2, x1:x2]
            else:
                mask_crop = mask
            body_feats = _extract_from_array(mask_crop)
            if mask_crop.shape[:2] == img.shape[:2]:
                img = (img * (mask_crop[:, :, None] / 255.0)).astype(np.uint8)

    # 3. Augment / normalise
    img = (train_aug if augment else val_aug)(image=img)['image'].astype(np.float32)

    # 4. AUX = [age_norm, sex_binary, 14 body-geometry features] => 16-dim
    age   = row.get('age', np.nan)
    age_n = float(age) / 10.0 if pd.notna(age) else 0.0
    sex_b = 1.0 if str(row.get('sex', '')).lower() in ('m', 'male', 'bull') else 0.0
    aux   = np.concatenate([[age_n, sex_b], body_feats]).astype(np.float32)

    # 5. Log-space target + label noise (sigma=0.01 ~ +/-1% scale error)
    log_w  = np.log(float(row['weight_kg']))
    if augment:
        log_w += np.random.normal(0.0, 0.01)
    w_norm = (log_w - LOG_MEAN) / LOG_STD
    return img, aux, np.float32(w_norm)


def _mixup_batch(inputs, labels):
    # MixUp at batch level (alpha=0.2) - reduces overfitting for regression.
    imgs  = inputs['image']
    auxs  = inputs['aux']
    B     = tf.shape(imgs)[0]
    lam   = tf.random.uniform((), 0.2, 0.8)
    idx_s = tf.random.shuffle(tf.range(B))
    mixed = {
        'image': lam * imgs + (1.0 - lam) * tf.gather(imgs, idx_s),
        'aux':   lam * auxs + (1.0 - lam) * tf.gather(auxs, idx_s),
    }
    return mixed, lam * labels + (1.0 - lam) * tf.gather(labels, idx_s)


def make_tf_dataset(df_split, batch_size=32, shuffle=True, augment=True):
    rows = df_split.to_dict('records')
    if shuffle:
        import random; random.seed(SEED); random.shuffle(rows)

    def gen():
        for row in rows:
            try:
                img, aux, w = load_sample(row, augment)
                yield {'image': img, 'aux': aux}, w
            except Exception:
                pass

    ds = tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            {'image': tf.TensorSpec((IMG_SIZE, IMG_SIZE, 3), tf.float32),
             'aux':   tf.TensorSpec((AUX_DIM,), tf.float32)},
            tf.TensorSpec((), tf.float32),
        ),
    )
    if not augment:
        ds = ds.cache()
    ds = ds.batch(batch_size)
    if augment:
        ds = ds.map(_mixup_batch, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)


_img, _aux, _w = load_sample(train_df.iloc[0].to_dict(), augment=False)
_recon = np.exp(_w * LOG_STD + LOG_MEAN)
print(f'image: {_img.shape}  aux: {_aux.shape}  w_log_norm: {_w:.3f}')
print(f'Reconstruction check: {_recon:.1f} kg  (true: {train_df.iloc[0].weight_kg:.1f} kg)')
print(f'AUX_DIM={AUX_DIM}  [age/10, sex_binary, 14 body-geometry features]')
print(f'Body features sample: {_aux[2:].round(3)}')
_has_mask = sum(1 for s in train_df['file_name'] if (MASK_DIR / (Path(s).stem+'.png')).exists())
print(f'Masks available: {_has_mask:,}/{len(train_df):,}')
print('MixUp: enabled (alpha=0.2)  |  Label noise: sigma=0.01')

In [ ]:
def mape_metric(y_true, y_pred):
    yt = tf.exp(y_true * LOG_STD + LOG_MEAN)
    yp = tf.exp(y_pred * LOG_STD + LOG_MEAN)
    return tf.reduce_mean(tf.abs(yt - yp) / (yt + 1e-6)) * 100.0

def mae_kg_metric(y_true, y_pred):
    yt = tf.exp(y_true * LOG_STD + LOG_MEAN)
    yp = tf.exp(y_pred * LOG_STD + LOG_MEAN)
    return tf.reduce_mean(tf.abs(yt - yp))


def _head(x, aux_inp, units=(256, 64), drop=0.3):
    """Fusion head: backbone_pool + aux -> log-space scalar."""
    a = layers.Dense(64, activation='swish')(aux_inp)
    a = layers.BatchNormalization()(a)
    x = layers.Concatenate()([x, a])
    x = layers.BatchNormalization()(x)
    for u in units:
        x = layers.Dense(u, activation='swish')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(drop)(x)
    return layers.Dense(1, dtype='float32', name='weight')(x)


def build_effnetb0(img_size=IMG_SIZE, aux_dim=AUX_DIM):
    from tensorflow.keras.applications import EfficientNetV2B0
    inp_img = layers.Input((img_size, img_size, 3), name='image')
    inp_aux = layers.Input((aux_dim,), name='aux')
    base = EfficientNetV2B0(include_top=False, weights='imagenet', include_preprocessing=False)
    base.trainable = False
    x = layers.GlobalAveragePooling2D()(base(inp_img))
    x = layers.Dropout(0.3)(x)
    return Model([inp_img, inp_aux], _head(x, inp_aux, (256, 64)), name='effnetv2b0')


def build_effnets(img_size=IMG_SIZE, aux_dim=AUX_DIM):
    from tensorflow.keras.applications import EfficientNetV2S
    inp_img = layers.Input((img_size, img_size, 3), name='image')
    inp_aux = layers.Input((aux_dim,), name='aux')
    base = EfficientNetV2S(include_top=False, weights='imagenet', include_preprocessing=False)
    base.trainable = False
    x = layers.GlobalAveragePooling2D()(base(inp_img))
    x = layers.Dropout(0.3)(x)
    return Model([inp_img, inp_aux], _head(x, inp_aux, (512, 256, 64)), name='effnetv2s')


def build_effnetm(img_size=IMG_SIZE, aux_dim=AUX_DIM):
    try:
        from tensorflow.keras.applications import EfficientNetV2M
        inp_img = layers.Input((img_size, img_size, 3), name='image')
        inp_aux = layers.Input((aux_dim,), name='aux')
        base = EfficientNetV2M(include_top=False, weights='imagenet', include_preprocessing=False)
        base.trainable = False
        x = layers.GlobalAveragePooling2D()(base(inp_img))
        x = layers.Dropout(0.4)(x)
        return Model([inp_img, inp_aux], _head(x, inp_aux, (512, 256, 64)), name='effnetv2m')
    except Exception:
        return None


def build_convnext(img_size=IMG_SIZE, aux_dim=AUX_DIM):
    try:
        from tensorflow.keras.applications import ConvNeXtTiny
    except ImportError:
        return None
    inp_img = layers.Input((img_size, img_size, 3), name='image')
    inp_aux = layers.Input((aux_dim,), name='aux')
    base = ConvNeXtTiny(include_top=False, weights='imagenet')
    base.trainable = False
    x = layers.GlobalAveragePooling2D()(base(inp_img))
    a = layers.Dense(64, activation='gelu')(inp_aux)
    x = layers.Concatenate()([x, a])
    x = layers.LayerNormalization()(x)
    x = layers.Dense(256, activation='gelu')(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='gelu')(x)
    return Model([inp_img, inp_aux],
                 layers.Dense(1, dtype='float32', name='weight')(x),
                 name='convnext_tiny')


def build_mobilenet(img_size=224, aux_dim=AUX_DIM):
    from tensorflow.keras.applications import MobileNetV3Large
    inp_img = layers.Input((img_size, img_size, 3), name='image')
    inp_aux = layers.Input((aux_dim,), name='aux')
    base = MobileNetV3Large(include_top=False, weights='imagenet', minimalistic=False,
                            include_preprocessing=False)
    base.trainable = False
    x = layers.GlobalAveragePooling2D()(base(inp_img))
    return Model([inp_img, inp_aux], _head(x, inp_aux, (128,), drop=0.2),
                 name='mobilenetv3')




def build_convnext_small(img_size=IMG_SIZE, aux_dim=AUX_DIM):
    try:
        from tensorflow.keras.applications import ConvNeXtSmall
    except ImportError:
        return None
    inp_img = layers.Input((img_size, img_size, 3), name='image')
    inp_aux = layers.Input((aux_dim,), name='aux')
    base = ConvNeXtSmall(include_top=False, weights='imagenet')
    base.trainable = False
    x = layers.GlobalAveragePooling2D()(base(inp_img))
    a = layers.Dense(64, activation='gelu')(inp_aux)
    x = layers.Concatenate()([x, a])
    x = layers.LayerNormalization()(x)
    x = layers.Dense(256, activation='gelu')(x)
    x = layers.Dropout(0.35)(x)
    x = layers.Dense(64, activation='gelu')(x)
    return Model([inp_img, inp_aux],
                 layers.Dense(1, dtype='float32', name='weight')(x),
                 name='convnext_small')


print('Model builders ready (B0, S, M, ConvNeXt-Tiny, ConvNeXt-Small, MobileNetV3).')

In [ ]:
class WarmupCosine(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base_lr, total_steps, warmup_steps):
        self.base_lr = base_lr
        self.total_steps = total_steps
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        w = self.base_lr * (step / self.warmup_steps)
        c = self.base_lr * 0.5 * (1 + tf.cos(
            math.pi * (step - self.warmup_steps)
            / (self.total_steps - self.warmup_steps)
        ))
        return tf.where(step < self.warmup_steps, w, c)

    def get_config(self):
        return dict(base_lr=self.base_lr, total_steps=self.total_steps,
                    warmup_steps=self.warmup_steps)


def make_callbacks(name, patience=10):
    return [
        keras.callbacks.ModelCheckpoint(
            str(OUT / f'{name}_best.keras'),
            monitor='val_mape_metric', mode='min',
            save_best_only=True, verbose=0),
        keras.callbacks.EarlyStopping(
            monitor='val_mape_metric', mode='min', patience=patience,
            restore_best_weights=True),
        keras.callbacks.CSVLogger(str(OUT / f'{name}_log.csv')),
    ]


def train_phase(model, train_ds, val_ds, epochs, lr, phase,
                patience=8, n_train=None):
    n     = n_train or len(train_df)
    steps = epochs * math.ceil(n / BATCH_SIZE)
    sched = WarmupCosine(lr, steps, max(1, int(0.05 * steps)))
    model.compile(
        optimizer=keras.optimizers.AdamW(
            sched, weight_decay=1e-4, clipnorm=1.0),
        loss='mae',  # MAE in log-space ≈ direct MAPE minimisation
        metrics=[mape_metric, mae_kg_metric],
    )
    print(f'\n=== {phase} ===')
    return model.fit(
        train_ds, validation_data=val_ds, epochs=epochs,
        callbacks=make_callbacks(model.name, patience), verbose=1,
    )


def predict_tta(model, rows, n_tta=6):
    """Average n_tta predictions in log-space then convert to kg."""
    all_log_preds = []
    for t in range(n_tta):
        log_preds = []
        for i in range(0, len(rows), BATCH_SIZE):
            batch   = rows[i:i + BATCH_SIZE]
            samples = [load_sample(r, augment=(t > 0)) for r in batch]
            imgs    = np.stack([s[0] for s in samples])
            auxes   = np.stack([s[1] for s in samples])
            p = model.predict({'image': imgs, 'aux': auxes}, verbose=0).ravel()
            log_preds.extend(p.tolist())
        all_log_preds.append(log_preds)
    avg_log = np.mean(all_log_preds, axis=0)
    return np.exp(np.array(avg_log) * LOG_STD + LOG_MEAN)


print('Training utilities ready (log-MAE loss + TTA).')

In [ ]:
BATCH_SIZE = 64 if FAST_MODE else 32   # 64 halves step count; fine at 224px

train_ds = make_tf_dataset(train_df, BATCH_SIZE, shuffle=True,  augment=True)
val_ds   = make_tf_dataset(val_df,   BATCH_SIZE, shuffle=False, augment=False)
test_ds  = make_tf_dataset(test_df,  BATCH_SIZE, shuffle=False, augment=False)

print(f'Batch size    : {BATCH_SIZE}')
print(f'AUX_DIM       : {AUX_DIM}  (age, sex, 14 body-geometry features)')
print(f'Train batches : {math.ceil(len(train_df)/BATCH_SIZE)}')
print(f'Val batches   : {math.ceil(len(val_df)/BATCH_SIZE)}')
print(f'Test batches  : {math.ceil(len(test_df)/BATCH_SIZE)}')
print('Val/test datasets cached after first epoch.')


In [ ]:
# ── V1: EfficientNetV2-B0, frozen backbone ─────────────────────────────────────
_v1_p = _ckpt('effnetv2b0')
if _v1_p:
    print(f'[RESUME] effnetv2b0 checkpoint found at {_v1_p} — skipping training')
    model_v1 = keras.models.load_model(str(_v1_p), compile=False)
    _v1_skip = True
else:
    model_v1 = build_effnetb0()
    model_v1.summary(line_length=90, expand_nested=False)
    hist_v1 = train_phase(
        model_v1, train_ds, val_ds,
        epochs=6 if FAST_MODE else 15, lr=1e-3,
        phase='V1 EfficientNetV2-B0 frozen',
        patience=3 if FAST_MODE else 5,
    )
    _v1_skip = False


In [ ]:
if _v1_skip:
    _r = _load_results('effnetv2b0')
    if _r:
        results_v1 = _r
        print(f'[RESUME] V1  MAPE={results_v1[1]:.2f}%  MAE={results_v1[2]:.1f} kg (cached)')
    else:
        model_v1.compile(optimizer='adam', loss='mae', metrics=[mape_metric, mae_kg_metric])
        results_v1 = model_v1.evaluate(test_ds, verbose=0)
        print(f'V1 Test  MAPE={results_v1[1]:.2f}%  MAE={results_v1[2]:.1f} kg')
        _save_results('effnetv2b0', results_v1)
else:
    for layer in model_v1.layers:
        if hasattr(layer, 'layers'):
            n = len(layer.layers)
            for sub in layer.layers[n // 2:]:
                sub.trainable = True

    hist_v1_ft = train_phase(
        model_v1, train_ds, val_ds,
        epochs=10 if FAST_MODE else 20, lr=5e-5,
        phase='V1 fine-tune',
        patience=4 if FAST_MODE else 7,
    )

    results_v1 = model_v1.evaluate(test_ds, verbose=0)
    print(f'V1 Test  MAPE={results_v1[1]:.2f}%  MAE={results_v1[2]:.1f} kg')
    _save_results('effnetv2b0', results_v1)


In [ ]:
# ── V2: EfficientNetV2-S, frozen backbone ──────────────────────────────────────
_v2_p = _ckpt('effnetv2s')
if _v2_p:
    print(f'[RESUME] effnetv2s checkpoint found at {_v2_p} — skipping training')
    model_v2 = keras.models.load_model(str(_v2_p), compile=False)
    _v2_skip = True
else:
    model_v2 = build_effnets()
    hist_v2 = train_phase(
        model_v2, train_ds, val_ds,
        epochs=6 if FAST_MODE else 12, lr=1e-3,
        phase='V2 EfficientNetV2-S frozen',
        patience=3 if FAST_MODE else 5,
    )
    _v2_skip = False


In [ ]:
if _v2_skip:
    _r = _load_results('effnetv2s')
    if _r:
        results_v2 = _r
        print(f'[RESUME] V2  MAPE={results_v2[1]:.2f}%  MAE={results_v2[2]:.1f} kg (cached)')
    else:
        model_v2.compile(optimizer='adam', loss='mae', metrics=[mape_metric, mae_kg_metric])
        results_v2 = model_v2.evaluate(test_ds, verbose=0)
        print(f'V2 Test  MAPE={results_v2[1]:.2f}%  MAE={results_v2[2]:.1f} kg')
        _save_results('effnetv2s', results_v2)
else:
    for layer in model_v2.layers:
        if hasattr(layer, 'layers'):
            n = len(layer.layers)
            for sub in layer.layers[n * 2 // 3:]:
                sub.trainable = True

    hist_v2_ft = train_phase(
        model_v2, train_ds, val_ds,
        epochs=12 if FAST_MODE else 25, lr=2e-5,
        phase='V2 fine-tune',
        patience=4 if FAST_MODE else 8,
    )

    results_v2 = model_v2.evaluate(test_ds, verbose=0)
    print(f'V2 Test  MAPE={results_v2[1]:.2f}%  MAE={results_v2[2]:.1f} kg')
    _save_results('effnetv2s', results_v2)


In [ ]:
# ── V3: ConvNeXt-Tiny ─────────────────────────────────────────────────────────
results_v3 = None
model_v3   = None
if FAST_MODE:
    print('[FAST_MODE] Skipping V3 ConvNeXt-Tiny')
else:
    _v3_p = _ckpt('convnext_tiny')
    if _v3_p:
        print(f'[RESUME] convnext_tiny found at {_v3_p}')
        model_v3 = keras.models.load_model(str(_v3_p), compile=False)
        _r = _load_results('convnext_tiny')
        if _r:
            results_v3 = _r
            print(f'[RESUME] V3  MAPE={results_v3[1]:.2f}%  MAE={results_v3[2]:.1f} kg')
        else:
            model_v3.compile(optimizer='adam', loss='mae', metrics=[mape_metric, mae_kg_metric])
            results_v3 = model_v3.evaluate(test_ds, verbose=0)
            _save_results('convnext_tiny', results_v3)
    else:
        model_v3 = build_convnext()
        if model_v3 is not None:
            hist_v3 = train_phase(model_v3, train_ds, val_ds,
                                  epochs=15, lr=1e-3,
                                  phase='V3 ConvNeXt-Tiny frozen', patience=5)
            for layer in model_v3.layers:
                if hasattr(layer, 'layers'):
                    for sub in layer.layers[-20:]: sub.trainable = True
            hist_v3_ft = train_phase(model_v3, train_ds, val_ds,
                                     epochs=15, lr=1e-5,
                                     phase='V3 fine-tune', patience=6)
            results_v3 = model_v3.evaluate(test_ds, verbose=0)
            print(f'V3 Test  MAPE={results_v3[1]:.2f}%  MAE={results_v3[2]:.1f} kg')
            _save_results('convnext_tiny', results_v3)
        else:
            print('ConvNeXt not available in this TF version.')


In [ ]:
# -- V4: ConvNeXt-Small --------------------------------------------------------
results_v4 = None
model_v4   = None
if FAST_MODE:
    print('[FAST_MODE] Skipping V4 ConvNeXt-Small')
else:
    _v4_p = _ckpt('convnext_small')
    if _v4_p:
        print(f'[RESUME] convnext_small found at {_v4_p}')
        model_v4 = keras.models.load_model(str(_v4_p), compile=False)
        _r = _load_results('convnext_small')
        if _r:
            results_v4 = _r
            print(f'[RESUME] V4  MAPE={results_v4[1]:.2f}%  MAE={results_v4[2]:.1f} kg')
        else:
            model_v4.compile(optimizer='adam', loss='mae', metrics=[mape_metric, mae_kg_metric])
            results_v4 = model_v4.evaluate(test_ds, verbose=0)
            _save_results('convnext_small', results_v4)
    else:
        model_v4 = build_convnext_small()
        if model_v4 is not None:
            hist_v4 = train_phase(model_v4, train_ds, val_ds,
                                  epochs=12, lr=1e-3,
                                  phase='V4 ConvNeXt-Small frozen', patience=5)
            for layer in model_v4.layers:
                if hasattr(layer, 'layers'):
                    for sub in layer.layers[-30:]: sub.trainable = True
            hist_v4_ft = train_phase(model_v4, train_ds, val_ds,
                                     epochs=20, lr=5e-6,
                                     phase='V4 ConvNeXt-Small fine-tune', patience=7)
            results_v4 = model_v4.evaluate(test_ds, verbose=0)
            print(f'V4 Test  MAPE={results_v4[1]:.2f}%  MAE={results_v4[2]:.1f} kg')
            _save_results('convnext_small', results_v4)
        else:
            print('ConvNeXt-Small not available.')


In [ ]:
# ── VM: EfficientNetV2-M ──────────────────────────────────────────────────────
results_vm = None
model_vm   = None
if FAST_MODE:
    print('[FAST_MODE] Skipping VM EfficientNetV2-M')
else:
    _vm_p = _ckpt('effnetv2m')
    if _vm_p:
        print(f'[RESUME] effnetv2m found at {_vm_p}')
        model_vm = keras.models.load_model(str(_vm_p), compile=False)
        _r = _load_results('effnetv2m')
        if _r:
            results_vm = _r
            print(f'[RESUME] VM  MAPE={results_vm[1]:.2f}%  MAE={results_vm[2]:.1f} kg')
        else:
            model_vm.compile(optimizer='adam', loss='mae', metrics=[mape_metric, mae_kg_metric])
            results_vm = model_vm.evaluate(test_ds, verbose=0)
            _save_results('effnetv2m', results_vm)
    else:
        model_vm = build_effnetm()
        if model_vm is not None:
            hist_vm = train_phase(model_vm, train_ds, val_ds,
                                  epochs=12, lr=1e-3,
                                  phase='VM EfficientNetV2-M frozen', patience=5)
            for layer in model_vm.layers:
                if hasattr(layer, 'layers'):
                    n = len(layer.layers)
                    for sub in layer.layers[n * 2 // 3:]: sub.trainable = True
            hist_vm_ft = train_phase(model_vm, train_ds, val_ds,
                                     epochs=25, lr=1e-5,
                                     phase='VM EfficientNetV2-M fine-tune', patience=8)
            results_vm = model_vm.evaluate(test_ds, verbose=0)
            print(f'VM Test  MAPE={results_vm[1]:.2f}%  MAE={results_vm[2]:.1f} kg')
            _save_results('effnetv2m', results_vm)
        else:
            print('EfficientNetV2-M not available.')


In [ ]:
# -- Pick best, full fine-tune all layers ----------------------------------
candidates = [
    ('V1_EfficientNetV2-B0', results_v1, model_v1),
    ('V2_EfficientNetV2-S',  results_v2, model_v2),
]
if results_v3 is not None:
    candidates.append(('V3_ConvNeXt-Tiny',  results_v3, model_v3))
if 'results_v4' in dir() and results_v4 is not None:
    candidates.append(('V4_ConvNeXt-Small', results_v4, model_v4))
if results_vm is not None:
    candidates.append(('VM_EfficientNetV2-M', results_vm, model_vm))

best_name, best_res, best_model = min(candidates, key=lambda x: x[1][1])
print(f'Best so far: {best_name}  MAPE={best_res[1]:.2f}%  MAE={best_res[2]:.1f} kg')

for cname, cres, cmodel in candidates:
    cmodel.save(str(OUT / f'{cname}_candidate.keras'))
    print(f'Saved: {cname}  MAPE={cres[1]:.2f}%')

_ft_p = _ckpt('cattle_weight_best')
if _ft_p:
    print(f'[RESUME] Full fine-tune checkpoint found at {_ft_p} — loading')
    best_model = keras.models.load_model(str(_ft_p), compile=False)
    _r = _load_results('cattle_weight_best')
    if _r:
        results_full = _r
        print(f'[RESUME] Full FT  MAPE={results_full[1]:.2f}%  MAE={results_full[2]:.1f} kg (cached)')
    else:
        best_model.compile(optimizer='adam', loss='mae', metrics=[mape_metric, mae_kg_metric])
        results_full = best_model.evaluate(test_ds, verbose=0)
        _save_results('cattle_weight_best', results_full)
    hist_full = type('H', (), {'history': {'mape_metric': [], 'val_mape_metric': []}})() 
else:
    for layer in best_model.layers:
        layer.trainable = True

    hist_full = train_phase(
        best_model, train_ds, val_ds,
        epochs=15 if FAST_MODE else 40, lr=5e-6,
        phase=f'{best_name} full fine-tune',
        patience=5 if FAST_MODE else 12,
    )

    results_full = best_model.evaluate(test_ds, verbose=0)
    print(f'After full fine-tune  MAPE={results_full[1]:.2f}%  MAE={results_full[2]:.1f} kg')
    best_model.save(str(OUT / 'cattle_weight_best_best.keras'))
    _save_results('cattle_weight_best', results_full)


In [ ]:
# -- TTA + MAPE-weighted multi-model ensemble ------------------------------
test_rows = test_df.to_dict('records')
y_true    = np.array([float(r['weight_kg']) for r in test_rows])
paths     = [r['full_path'] for r in test_rows]

print('Loading all candidate models for ensemble...')
ckpt_preds_list = []
ckpt_mapes      = []

for ckpt in sorted(OUT.glob('*.keras')):
    if 'mobile' in ckpt.name.lower():
        continue
    try:
        m = keras.models.load_model(str(ckpt), compile=False)
        preds_kg = predict_tta(m, test_rows, n_tta=6)
        mape_ck  = np.mean(np.abs(preds_kg - y_true) / y_true) * 100
        ckpt_preds_list.append(preds_kg)
        ckpt_mapes.append(mape_ck)
        print(f'  {ckpt.stem}: MAPE={mape_ck:.2f}%')
        del m; gc.collect()
    except Exception as e:
        print(f'  Skip {ckpt.name}: {e}')

if not ckpt_preds_list:
    ckpt_preds_list.append(predict_tta(best_model, test_rows, n_tta=6))
    ckpt_mapes.append(float(results_full[1]))

# Weight each model by 1/MAPE so better models contribute more
weights  = np.array([1.0 / (m + 1e-6) for m in ckpt_mapes])
weights /= weights.sum()
print('Ensemble weights (1/MAPE): ' +
      '  '.join(f'm{i}={w:.3f}' for i, w in enumerate(weights)))
y_pred   = np.average(np.stack(ckpt_preds_list, axis=0), axis=0, weights=weights)

abs_err = np.abs(y_true - y_pred)
pct_err = abs_err / y_true * 100
mape_f  = pct_err.mean()
mae_f   = abs_err.mean()
rmse_f  = np.sqrt(((y_true - y_pred)**2).mean())
r2_f    = 1 - ((y_true - y_pred)**2).sum() / ((y_true - y_true.mean())**2).sum()

print(f'\nEnsemble of {len(ckpt_preds_list)} models + 6x TTA (MAPE-weighted)')
print('=' * 50)
print('FINAL TEST METRICS')
print(f'  MAPE : {mape_f:.2f}%')
print(f'  MAE  : {mae_f:.1f} kg')
print(f'  RMSE : {rmse_f:.1f} kg')
print(f'  R2   : {r2_f:.4f}')
print(f'  <5%  : {(pct_err<5).mean()*100:.1f}%  |  <8%: {(pct_err<8).mean()*100:.1f}%')
print('=' * 50)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

ax = axes[0, 0]
ax.scatter(y_true, y_pred, alpha=0.35, s=12)
mn, mx = min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())
ax.plot([mn, mx], [mn, mx], 'r--')
ax.set(xlabel='True (kg)', ylabel='Predicted (kg)',
       title=f'True vs Predicted  MAPE={mape_f:.2f}%')

ax = axes[0, 1]
ax.hist(pct_err, bins=40, color='steelblue', edgecolor='white')
ax.axvline(mape_f, color='r', ls='--', label=f'Mean {mape_f:.1f}%')
ax.axvline(5, color='g', ls=':', label='5% target')
ax.axvline(8, color='orange', ls=':', label='8% target')
ax.legend()
ax.set(title='% Error Distribution', xlabel='% Error')

ax = axes[0, 2]
ax.scatter(y_true, y_pred - y_true, alpha=0.35, s=12)
ax.axhline(0, color='r', ls='--')
ax.set(xlabel='True (kg)', ylabel='Residual (kg)', title='Residuals')

ax = axes[1, 0]
qlab = pd.qcut(y_true, q=5, labels=['Q1', 'Q2', 'Q3', 'Q4', 'Q5'])
pd.DataFrame({'q': qlab, 'e': pct_err}).boxplot(column='e', by='q', ax=ax)
plt.sca(ax); plt.title('% Error by Weight Quantile')
ax.set_ylabel('% Error')

ax = axes[1, 1]
s = np.sort(pct_err)
ax.plot(s, np.arange(1, len(s)+1) / len(s) * 100)
ax.axvline(5, color='g', ls=':', label=f'<5%: {(pct_err<5).mean()*100:.0f}%')
ax.axvline(8, color='orange', ls=':', label=f'<8%: {(pct_err<8).mean()*100:.0f}%')
ax.legend()
ax.set(xlabel='% Error', ylabel='Cumulative %', title='Cumulative Error')

ax = axes[1, 2]
h = hist_full.history
ax.plot(h['mape_metric'], label='Train')
ax.plot(h['val_mape_metric'], label='Val')
ax.legend()
ax.set(xlabel='Epoch', ylabel='MAPE %', title='Learning Curve (full fine-tune)')

plt.suptitle(f'Error Analysis  —  {best_name}')
plt.tight_layout()
plt.savefig(str(OUT / 'error_analysis.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
worst = np.argsort(pct_err)[-10:][::-1]
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for ax, idx in zip(axes.flat, worst):
    try:
        img = Image.open(paths[idx]).convert('RGB')
        img.thumbnail((300, 300))
        ax.imshow(img)
        ax.set_title(
            f'True: {y_true[idx]:.0f} kg\nPred: {y_pred[idx]:.0f} kg\nErr: {pct_err[idx]:.1f}%',
            fontsize=9, color='red')
    except Exception:
        ax.text(0.5, 0.5, 'load error', ha='center', transform=ax.transAxes)
    ax.axis('off')
plt.suptitle('Worst 10 Predictions', fontsize=13)
plt.tight_layout()
plt.savefig(str(OUT / 'worst_predictions.png'), dpi=100)
plt.show()

In [ ]:
import json as _json
_json.dump({
    'weight_mean': WEIGHT_MEAN, 'weight_std': WEIGHT_STD,
    'log_mean': LOG_MEAN, 'log_std': LOG_STD,
    'img_size': IMG_SIZE, 'aux_dim': AUX_DIM,
    'img_mean': _MEAN, 'img_std': _STD,
    'mape_test': float(mape_f), 'mae_test_kg': float(mae_f),
    'normalization': 'log',
}, open(OUT / 'normalization_info.json', 'w'), indent=2)
print('Normalization info saved (log-space).')

# FP16
conv_fp16 = tf.lite.TFLiteConverter.from_keras_model(best_model)
conv_fp16.optimizations = [tf.lite.Optimize.DEFAULT]
conv_fp16.target_spec.supported_types = [tf.float16]
tflite_fp16 = conv_fp16.convert()
(OUT / 'cattle_weight_fp16.tflite').write_bytes(tflite_fp16)
print(f'FP16 TFLite: {len(tflite_fp16)/1e6:.2f} MB')

# INT8
def _rep_ds():
    for row in val_df.head(100).to_dict('records'):
        try:
            img, aux, _ = load_sample(row, augment=False)
            yield [img[np.newaxis].astype(np.float32),
                   aux[np.newaxis].astype(np.float32)]
        except Exception:
            pass

conv_int8 = tf.lite.TFLiteConverter.from_keras_model(best_model)
conv_int8.optimizations = [tf.lite.Optimize.DEFAULT]
conv_int8.representative_dataset = _rep_ds
conv_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv_int8.inference_input_type  = tf.float32
conv_int8.inference_output_type = tf.float32
try:
    tflite_int8 = conv_int8.convert()
    (OUT / 'cattle_weight_int8.tflite').write_bytes(tflite_int8)
    print(f'INT8 TFLite: {len(tflite_int8)/1e6:.2f} MB')
except Exception as e:
    print(f'INT8 failed: {e} — using FP16 as fallback')
    tflite_int8 = tflite_fp16

In [ ]:
interp = tf.lite.Interpreter(model_content=tflite_fp16)
interp.allocate_tensors()
for det in interp.get_input_details():
    interp.set_tensor(det['index'], np.zeros(det['shape'], np.float32))
for _ in range(5):
    interp.invoke()
t0 = time.time()
for _ in range(50):
    interp.invoke()
ms = (time.time() - t0) / 50 * 1000
print(f'FP16 TFLite latency (Kaggle CPU): {ms:.1f} ms/inference')
print(f'Expected on Android ARM ~{ms*2:.0f} ms  (target <300 ms)')

In [ ]:
# ── MobileNetV3-Large: lightweight on-device model ────────────────────────────
MOBILE_SIZE = 224

_aug_m = A.Compose([
    A.LongestMaxSize(MOBILE_SIZE),
    A.PadIfNeeded(MOBILE_SIZE, MOBILE_SIZE, border_mode=cv2.BORDER_CONSTANT, value=0),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(0.2, 0.2, p=0.5),
    A.Normalize(_MEAN, _STD),
])
_aug_m_val = A.Compose([
    A.LongestMaxSize(MOBILE_SIZE),
    A.PadIfNeeded(MOBILE_SIZE, MOBILE_SIZE, border_mode=cv2.BORDER_CONSTANT, value=0),
    A.Normalize(_MEAN, _STD),
])

_aug_orig, _aug_val_orig, _img_size_orig = train_aug, val_aug, IMG_SIZE
train_aug, val_aug, IMG_SIZE = _aug_m, _aug_m_val, MOBILE_SIZE

train_ds_m = make_tf_dataset(train_df, BATCH_SIZE, shuffle=True,  augment=True)
val_ds_m   = make_tf_dataset(val_df,   BATCH_SIZE, shuffle=False, augment=False)

_mob_p = _ckpt('mobilenetv3')
if _mob_p:
    print(f'[RESUME] mobilenetv3 checkpoint found at {_mob_p} — skipping training')
    student = keras.models.load_model(str(_mob_p), compile=False)
    _r = _load_results('mobilenetv3')
    if _r:
        res_mob = _r
        print(f'[RESUME] MobileNetV3 Val  MAPE={res_mob[1]:.2f}%  MAE={res_mob[2]:.1f} kg (cached)')
    else:
        student.compile(optimizer='adam', loss='mae', metrics=[mape_metric, mae_kg_metric])
        res_mob = student.evaluate(val_ds_m, verbose=0)
        _save_results('mobilenetv3', res_mob)
else:
    student = build_mobilenet(img_size=MOBILE_SIZE)
    student.summary(line_length=80, expand_nested=False)

    hist_mob = train_phase(
        student, train_ds_m, val_ds_m,
        epochs=10 if FAST_MODE else 25, lr=1e-3,
        phase='MobileNetV3 frozen',
        patience=3 if FAST_MODE else 6,
        n_train=len(train_df),
    )

    for layer in student.layers:
        if hasattr(layer, 'layers'):
            n = len(layer.layers)
            for sub in layer.layers[n // 2:]: sub.trainable = True

    hist_mob_ft = train_phase(
        student, train_ds_m, val_ds_m,
        epochs=8 if FAST_MODE else 20, lr=5e-5,
        phase='MobileNetV3 fine-tune',
        patience=3 if FAST_MODE else 7,
        n_train=len(train_df),
    )

    res_mob = student.evaluate(val_ds_m, verbose=0)
    print(f'MobileNetV3 Val  MAPE={res_mob[1]:.2f}%  MAE={res_mob[2]:.1f} kg')
    student.save(str(OUT / 'cattle_weight_mobile.keras'))
    _save_results('mobilenetv3', res_mob)

train_aug, val_aug, IMG_SIZE = _aug_orig, _aug_val_orig, _img_size_orig


In [ ]:
conv_m = tf.lite.TFLiteConverter.from_keras_model(student)
conv_m.optimizations = [tf.lite.Optimize.DEFAULT]
conv_m.target_spec.supported_types = [tf.float16]
mob_tflite = conv_m.convert()
(OUT / 'cattle_weight_mobile_fp16.tflite').write_bytes(mob_tflite)
print(f'Mobile TFLite: {len(mob_tflite)/1e6:.2f} MB  (target <30 MB)')

interp_m = tf.lite.Interpreter(model_content=mob_tflite)
interp_m.allocate_tensors()
for det in interp_m.get_input_details():
    interp_m.set_tensor(det['index'], np.zeros(det['shape'], np.float32))
for _ in range(5):
    interp_m.invoke()
t0 = time.time()
for _ in range(50):
    interp_m.invoke()
ms_m = (time.time() - t0) / 50 * 1000
print(f'Mobile latency (CPU): {ms_m:.1f} ms  ->  Android ~{ms_m*2:.0f} ms')

In [ ]:
print('=' * 65)
print('ARCHITECTURE COMPARISON')
print('=' * 65)
IMG_SIZE_ORIG = _img_size_orig if '_img_size_orig' in dir() else IMG_SIZE

rows = [
    {'Model': 'V1 EfficientNetV2-B0',
     'MAPE%': f'{results_v1[1]:.2f}', 'MAE kg': f'{results_v1[2]:.1f}',
     'Input': f'{IMG_SIZE_ORIG}x{IMG_SIZE_ORIG}x3'},
    {'Model': 'V2 EfficientNetV2-S',
     'MAPE%': f'{results_v2[1]:.2f}', 'MAE kg': f'{results_v2[2]:.1f}',
     'Input': f'{IMG_SIZE_ORIG}x{IMG_SIZE_ORIG}x3'},
]
if results_v3 is not None:
    rows.append({'Model': 'V3 ConvNeXt-Tiny',
                 'MAPE%': f'{results_v3[1]:.2f}', 'MAE kg': f'{results_v3[2]:.1f}',
                 'Input': f'{IMG_SIZE_ORIG}x{IMG_SIZE_ORIG}x3'})
if 'results_v4' in dir() and results_v4 is not None:
    rows.append({'Model': 'V4 ConvNeXt-Small',
                 'MAPE%': f'{results_v4[1]:.2f}', 'MAE kg': f'{results_v4[2]:.1f}',
                 'Input': f'{IMG_SIZE_ORIG}x{IMG_SIZE_ORIG}x3'})
if results_vm is not None:
    rows.append({'Model': 'VM EfficientNetV2-M',
                 'MAPE%': f'{results_vm[1]:.2f}', 'MAE kg': f'{results_vm[2]:.1f}',
                 'Input': f'{IMG_SIZE_ORIG}x{IMG_SIZE_ORIG}x3'})
rows.append({'Model': f'{best_name} (full FT)',
             'MAPE%': f'{results_full[1]:.2f}', 'MAE kg': f'{results_full[2]:.1f}',
             'Input': f'{IMG_SIZE_ORIG}x{IMG_SIZE_ORIG}x3'})
rows.append({'Model': f'Ensemble+TTA MAPE-weighted ({len(ckpt_preds_list)} models)',
             'MAPE%': f'{mape_f:.2f}', 'MAE kg': f'{mae_f:.1f}',
             'Input': f'{IMG_SIZE_ORIG}x{IMG_SIZE_ORIG}x3'})
rows.append({'Model': 'MobileNetV3 (deploy)',
             'MAPE%': f'{res_mob[1]:.2f}', 'MAE kg': f'{res_mob[2]:.1f}',
             'Input': f'{MOBILE_SIZE}x{MOBILE_SIZE}x3'})

print(pd.DataFrame(rows).to_string(index=False))

print('\nOutputs in /kaggle/working/:')
for f in sorted(OUT.glob('*')):
    if f.is_file():
        print(f'  {f.name}: {f.stat().st_size/1e6:.2f} MB')